# CLOVA Studio 검증 노트북

HyperCLOVA X 연결을 검증하고, **예산·설계에 필요한 미지수를 실측**한다.
결과는 [clova_budget.ipynb](clova_budget.ipynb) 의 CONFIG로 그대로 옮긴다.

| § | 측정 | 왜 필요한가 |
|---|---|---|
| 1 | 엔드포인트·인증 자동 탐색 | API 경로가 바뀌어도 노트북이 살아남게 |
| 2 | 기본 호출 + `usage` | 토큰 회계의 기준 |
| 3 | **한국어 문자/토큰 비율** | 코퍼스 535M 토큰 추정의 핵심 계수 |
| 4 | **지연시간** | 평가에서 속도가 기준이 될 수 있음 |
| 5 | **CoT 작동 여부** | Phase 1 게이트 — 안 되면 뼈대가 바뀐다 |
| 6 | function calling 지원 | 주최 측 참조 아키텍처의 전제 |
| 7 | CONFIG 출력 | 예산 노트북에 붙여넣기 |

## 사전 준비 (재현을 위해 반드시)

```bash
# 저장소 루트에서
cp .env.example .env
# .env 를 열어 CLOVA_API_KEY 값을 채운다 (키는 발급 시 1회만 평문 노출)
```

`.env` 는 `.gitignore` 에 등록되어 있다. **키를 노트북에 직접 쓰지 말 것.**

의존성: 표준 라이브러리만 사용한다 (`urllib`). 별도 설치 불필요.

In [1]:
# ============================================================
# 설정 — 키는 .env 에서만 읽는다
# ============================================================
import json, os, re, statistics, sys, time, urllib.error, urllib.request
from pathlib import Path

def find_repo_root(marker='.env.example'):
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / marker).exists() or (base / '.git').exists():
            return base
    return here

ROOT = find_repo_root()

def load_env(path=None):
    p = Path(path) if path else ROOT / '.env'
    if not p.exists():
        return False
    for line in p.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
    return True

loaded = load_env()
API_KEY = os.environ.get('CLOVA_API_KEY', '')

HOST = 'https://clovastudio.stream.ntruss.com'
MODEL = 'HCX-005'
TIMEOUT = 60

print('repo root :', ROOT)
print('.env      :', 'loaded' if loaded else 'NOT FOUND — .env.example 를 복사해 만드세요')
print('API key   :', (API_KEY[:6] + '...' + API_KEY[-4:]) if API_KEY else '없음')
assert API_KEY, 'CLOVA_API_KEY 가 없습니다. .env 를 확인하세요.'

repo root : C:\Users\Juhwankim\Documents\Github\ARGUS
.env      : loaded
API key   : nv-509...SeSF


---
## 1. 엔드포인트·인증 자동 탐색

CLOVA Studio는 **모델마다 지원 API 버전이 다르다** — HCX-005는 `v3`가 필요하고
`v1`은 `40084 Unsupported API for model` 을 돌려준다. 또 `nv-` 로 시작하는 신규 키는
레거시 헤더(`X-NCP-CLOVASTUDIO-API-KEY`)를 거부한다(`40104`).

경로가 바뀌어도 노트북이 죽지 않도록 **조합을 탐색해 살아있는 것을 찾는다.**

In [2]:
def _post(url, headers, body, timeout=TIMEOUT):
    req = urllib.request.Request(
        url, data=json.dumps(body).encode('utf-8'), method='POST',
        headers={'Content-Type': 'application/json', 'Accept': 'application/json', **headers})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.loads(r.read().decode('utf-8')), time.time() - t0
    except urllib.error.HTTPError as e:
        raw = e.read().decode('utf-8', 'replace')
        try:
            return e.code, json.loads(raw), time.time() - t0
        except Exception:
            return e.code, {'raw': raw}, time.time() - t0
    except Exception as e:
        return None, {'error': f'{type(e).__name__}: {e}'}, time.time() - t0

CANDIDATE_PATHS = [
    '/testapp/v3/chat-completions/{m}',
    '/v3/chat-completions/{m}',
    '/testapp/v1/chat-completions/{m}',
    '/v1/chat-completions/{m}',
]
CANDIDATE_AUTHS = [
    ('Bearer',                    lambda k: {'Authorization': f'Bearer {k}'}),
    ('X-NCP-CLOVASTUDIO-API-KEY', lambda k: {'X-NCP-CLOVASTUDIO-API-KEY': k}),
]
PROBE_BODY = {'messages': [{'role': 'user', 'content': '안녕하세요'}],
              'maxTokens': 10, 'temperature': 0.0, 'topP': 0.8}

def discover(model=MODEL, verbose=True):
    for path in CANDIDATE_PATHS:
        for aname, afn in CANDIDATE_AUTHS:
            url = HOST + path.format(m=model)
            st, body, dt = _post(url, afn(API_KEY), PROBE_BODY, timeout=30)
            msg = (body.get('status') or {}).get('message', body.get('error', ''))
            if verbose:
                print(f'[{str(st):>4}] {path.format(m=model):<40} {aname:<26} {dt:5.2f}s  {str(msg)[:70]}')
            if st == 200:
                return {'url': url, 'auth_name': aname, 'headers': afn(API_KEY)}
    return None

ENDPOINT = discover()
assert ENDPOINT, '작동하는 엔드포인트를 찾지 못했습니다.'
print()
print('>>> URL :', ENDPOINT['url'])
print('>>> AUTH:', ENDPOINT['auth_name'])

[ 200] /testapp/v3/chat-completions/HCX-005     Bearer                      0.69s  OK

>>> URL : https://clovastudio.stream.ntruss.com/testapp/v3/chat-completions/HCX-005
>>> AUTH: Bearer


---
## 2. 호출 헬퍼 + `usage` 확인

모든 호출을 이 함수로 통일하고 **토큰·지연을 항상 기록**한다.
이 로깅이 없으면 예산 관리가 불가능하다.

In [3]:
CALL_LOG = []   # 이 노트북이 쓴 모든 호출 기록

def chat(messages, max_tokens=256, temperature=0.0, top_p=0.8,
         repeat_penalty=1.1, tools=None, tag=''):
    body = {'messages': messages, 'temperature': temperature,
            'topP': top_p, 'repeatPenalty': repeat_penalty}
    if tools:
        # v3에서 tools 와 maxTokens 는 동시 지정 불가 (40001 conflict)
        body['tools'] = tools
        body['toolChoice'] = 'auto'
    else:
        body['maxTokens'] = max_tokens
    st, resp, dt = _post(ENDPOINT['url'], ENDPOINT['headers'], body)
    res = (resp or {}).get('result') or {}
    usage = res.get('usage') or {}
    rec = {
        'tag': tag, 'status': st, 'latency_s': dt,
        'prompt_tokens': usage.get('promptTokens'),
        'completion_tokens': usage.get('completionTokens'),
        'total_tokens': usage.get('totalTokens'),
        'finish': res.get('finishReason'),
        'text': (res.get('message') or {}).get('content', ''),
        'raw': resp,
    }
    CALL_LOG.append(rec)
    return rec

r = chat([{'role': 'user', 'content': '공시 데이터 분석을 한 문장으로 설명해줘.'}],
         max_tokens=80, tag='smoke')
print('status  :', r['status'], '|', r['finish'])
print('latency :', f"{r['latency_s']:.2f}s")
print('tokens  : prompt', r['prompt_tokens'], '/ completion', r['completion_tokens'],
      '/ total', r['total_tokens'])
print('---')
print(r['text'])

status  : 200 | length
latency : 2.07s
tokens  : prompt 16 / completion 80 / total 96
---
공시 데이터 분석은 기업의 재무 상태, 경영 성과 및 주요 주주 정보 등을 포함한 공시 정보를 수집하고 이를 체계적으로 분석하여 투자자들이 합리적인 의사결정을 내릴 수 있도록 지원하는 과정입니다.

1. **재무제표 분석**: 손익계산서, 대차대조표, 현금흐름표를 통해 기업의 수익성, 안정성, 성장성을 평가


---
## 3. 한국어 문자/토큰 비율 실측 ★

`clova_budget.ipynb` 의 `CHARS_PER_TOKEN_KO` 는 코퍼스 토큰 수 추정 전체를 좌우하는
계수다(민감도 2위). **실제 공시 원문**으로 재야 의미가 있으므로,
코퍼스가 있으면 거기서 표본을 뽑고 없으면 내장 샘플을 쓴다.

측정 원리: 같은 텍스트를 보내고 응답의 `promptTokens` 에서 **오버헤드를 뺀다.**
오버헤드는 빈 메시지로 따로 잰다.

In [4]:
FALLBACK_SAMPLE = (
    '당사는 2025년 7월 28일 이사회 결의를 통해 반도체 위탁생산 공급계약 체결을 결정하였습니다. '
    '계약금액은 22,764,764,160,000원으로 최근 사업연도 매출액 대비 7.6%에 해당합니다. '
    '계약상대는 글로벌 대형기업이며, 계약기간은 2025년 7월 24일부터 2033년 12월 31일까지입니다. '
    '계약상대방의 요청에 따라 경영상 비밀유지를 사유로 일부 정보의 공시를 유보하였습니다.'
)

HANGUL = re.compile(r'[가-힣]')

def strip_markup(t):
    """CSS/스크립트 블록을 먼저 제거해야 한다 — DART XML은 <style>이 앞머리에 크게 붙어 있어
    단순 태그 제거만 하면 CSS 문자열을 본문으로 오인한다."""
    t = re.sub(r'(?is)<style[^>]*>.*?</style>', ' ', t)
    t = re.sub(r'(?is)<script[^>]*>.*?</script>', ' ', t)
    t = re.sub(r'(?s)<[^>]*>', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def hangul_ratio(t):
    return len(HANGUL.findall(t)) / len(t) if t else 0.0

def load_corpus_sample(target_chars=1200, min_hangul=0.30):
    base = ROOT / 'data' / 'dart-disclosures' / 'raw' / 'exchange'
    if not base.is_dir():
        return FALLBACK_SAMPLE, 'fallback(내장 샘플)'
    for f in sorted(base.rglob('*.xml'))[:80]:
        try:
            t = strip_markup(f.read_text(encoding='utf-8', errors='ignore'))
        except OSError:
            continue
        if len(t) < target_chars:
            continue
        # 한글 비중이 높은 구간을 찾는다 (앞머리는 여전히 서식 잔재일 수 있음)
        for start in range(0, min(len(t) - target_chars, 20000), 500):
            seg = t[start:start + target_chars]
            if hangul_ratio(seg) >= min_hangul:
                return seg, f'corpus({f.name} @{start})'
    return FALLBACK_SAMPLE, 'fallback(내장 샘플)'

SAMPLE, SAMPLE_SRC = load_corpus_sample()
print('표본 출처:', SAMPLE_SRC, '|', len(SAMPLE), '자')
print(SAMPLE[:160], '...')
print()

# 오버헤드 측정 (역할 토큰 등)
base_r = chat([{'role': 'user', 'content': ''}], max_tokens=1, tag='overhead')
OVERHEAD = base_r['prompt_tokens'] or 0
print('프롬프트 오버헤드:', OVERHEAD, '토큰')

meas = chat([{'role': 'user', 'content': SAMPLE}], max_tokens=1, tag='chars_per_token')
body_tokens = (meas['prompt_tokens'] or 0) - OVERHEAD
CHARS_PER_TOKEN_KO = len(SAMPLE) / body_tokens if body_tokens > 0 else float('nan')

print(f'본문 {len(SAMPLE)}자 = {body_tokens}토큰')
print(f'>>> CHARS_PER_TOKEN_KO = {CHARS_PER_TOKEN_KO:.3f}')
print(f'    (기존 가정값 1.4 대비 {CHARS_PER_TOKEN_KO/1.4:.2f}배)')

표본 출처: corpus(20241218800563.xml @0) | 1200 자
HD현대일렉트릭/단일판매ㆍ공급계약 체결(자율공시)/(2024.12.18)단일판매ㆍ공급계약 체결(자율공시) 정정신고(보고) 정정일자 2024-12-18 1. 정정관련 공시서류 단일판매ㆍ공급계약 체결(자율공시)(변압기 5대) 2. 정정관련 공시서류제출일 2024-08-30 3. 정정사유 자 ...

프롬프트 오버헤드: 5 토큰
본문 1200자 = 697토큰
>>> CHARS_PER_TOKEN_KO = 1.722
    (기존 가정값 1.4 대비 1.23배)


---
## 4. 지연시간 측정

평가에서 **응답 속도가 기준이 될 수 있다**([CONVENTION §4](../docs/CONVENTION.md)).
출력 길이별로 재서 "생성 토큰당 비용(시간)"을 파악한다.

In [5]:
LAT = []
for mt in (32, 128, 512):
    for i in range(3):
        r = chat([{'role': 'user', 'content': '공시 데이터에서 계약금액을 추출하는 방법을 설명해줘.'}],
                 max_tokens=mt, temperature=0.5, tag=f'lat_{mt}')
        LAT.append((mt, r['completion_tokens'] or 0, r['latency_s']))

print(f"{'maxTokens':>10}{'생성토큰(평균)':>16}{'지연(평균)':>12}{'지연(최대)':>12}{'초/토큰':>10}")
print('-' * 62)
for mt in (32, 128, 512):
    rows = [x for x in LAT if x[0] == mt]
    ct = statistics.mean(r[1] for r in rows)
    la = statistics.mean(r[2] for r in rows)
    mx = max(r[2] for r in rows)
    print(f'{mt:>10}{ct:>16.0f}{la:>12.2f}{mx:>12.2f}{(la/ct if ct else 0):>10.3f}')
print()
print('※ 질문당 호출을 N회 쌓으면 지연도 N배가 된다 — 파이프라인 호출 수 상한의 근거')

 maxTokens        생성토큰(평균)      지연(평균)      지연(최대)      초/토큰
--------------------------------------------------------------
        32              32        1.67        2.97     0.052
       128             128        3.11        3.13     0.024
       512             347        7.98        8.85     0.023

※ 질문당 호출을 N회 쌓으면 지연도 N배가 된다 — 파이프라인 호출 수 상한의 근거


---
## 5. CoT 작동 여부 ★ — Phase 1 게이트

[paper/base/base.md](../paper/base/base.md) §2 — CoT는 **창발 능력**이고
스케일이 부족하면 *"유창하지만 비논리적인"* 추론이 나와 **표준 프롬프팅보다 나빠진다.**
평가 8지표에 「근거 기반(환각)」이 있으므로 그대로 감점이다.

**가정하고 넘어가면 안 되는 항목**이라 여기서 A/B로 확인한다.
문항은 공시 도메인의 계산·비교 형태로 구성했고 정답이 고정되어 있다.

In [6]:
QUESTIONS = [
    dict(q='어떤 기업의 2025년 매출액이 3,336,059억원이고 그중 DS부문이 1,301,282억원이다. '
           'DS부문 비중은 몇 %인가? 소수점 첫째 자리까지.',
         a='39.0'),
    dict(q='계약금액이 22,764,764,160,000원이고 최근 사업연도 매출액이 300,870,903,000,000원이다. '
           '매출액 대비 계약금액 비율은 몇 %인가? 소수점 첫째 자리까지.',
         a='7.6'),
    dict(q='A사 설비투자는 11조 2천억원, B사는 8조 5천억원이다. A사가 B사보다 몇 조원 더 많은가? '
           '소수점 첫째 자리까지 조 단위로.',
         a='2.7'),
    dict(q='2024년 영업이익이 6조 5천억원, 2025년이 8조 1천억원이다. 증가율은 몇 %인가? '
           '소수점 첫째 자리까지.',
         a='24.6'),
    dict(q='계약기간이 2025년 7월 24일부터 2033년 12월 31일까지다. 계약기간은 몇 년 몇 개월인가? '
           '"N년 M개월" 형식으로만 답하라.',
         a='8년 5개월'),
    dict(q='분기 시설투자가 DS 10.2조원, SDC 0.6조원, 기타 0.5조원이다. 총 시설투자는 몇 조원인가? '
           '소수점 첫째 자리까지.',
         a='11.3'),
]

DIRECT = '숫자만 간결히 답하라. 설명하지 마라.'
COT = '단계적으로 계산 과정을 적은 뒤, 마지막 줄에 "정답: <값>" 형식으로 답하라.'

def grade(text, gold):
    if not text:
        return False
    t = text.replace(',', '').replace(' ', '')
    g = gold.replace(',', '').replace(' ', '')
    return g in t

results = {'direct': [], 'cot': []}
for i, item in enumerate(QUESTIONS):
    for mode, sysmsg, mt in (('direct', DIRECT, 48), ('cot', COT, 512)):
        r = chat([{'role': 'system', 'content': sysmsg},
                  {'role': 'user', 'content': item['q']}],
                 max_tokens=mt, temperature=0.0, tag=f'cot_{mode}')
        ok = grade(r['text'], item['a'])
        results[mode].append(ok)
        print(f"Q{i+1} {mode:<7} {'O' if ok else 'X'}  "
              f"gold={item['a']:<10} lat={r['latency_s']:4.1f}s "
              f"out={r['completion_tokens']:>4}tok  {r['text'][:60].replace(chr(10),' / ')}")
    print()

n = len(QUESTIONS)
d_acc = sum(results['direct']) / n * 100
c_acc = sum(results['cot']) / n * 100
d_tok = statistics.mean(r['completion_tokens'] or 0 for r in CALL_LOG if r['tag'] == 'cot_direct')
c_tok = statistics.mean(r['completion_tokens'] or 0 for r in CALL_LOG if r['tag'] == 'cot_cot')

print('=' * 58)
print(f"{'모드':<10}{'정확도':>10}{'평균 출력토큰':>16}")
print('-' * 58)
print(f"{'Direct':<10}{d_acc:>9.1f}%{d_tok:>16.0f}")
print(f"{'CoT':<10}{c_acc:>9.1f}%{c_tok:>16.0f}")
print('=' * 58)
delta = c_acc - d_acc
if delta > 10:
    print(f'판정: CoT 유효 (+{delta:.1f}%p). 난이도 상 질의에 CoT 경로를 쓴다.')
elif delta < -5:
    print(f'판정: CoT 역효과 ({delta:.1f}%p). base.md §2의 경고 사례 — 뼈대 재검토 필요.')
else:
    print(f'판정: 차이 미미 ({delta:+.1f}%p). 비용({c_tok/max(d_tok,1):.1f}배) 대비 이득을 재봐야 한다.')
print(f'비용 배수: CoT가 Direct의 {c_tok/max(d_tok,1):.1f}배 출력 토큰')

Q1 direct  X  gold=39.0       lat= 0.4s out=   5tok  38.9%
Q1 cot     O  gold=39.0       lat= 3.4s out= 139tok  DS 부문의 비중을 구하기 위해 다음과 같은 계산을 수행합니다. /  / 1. 전체 매출액에서 DS 부문이 차지하는

Q2 direct  O  gold=7.6        lat= 0.5s out=   5tok  7.6%
Q2 cot     O  gold=7.6        lat= 5.5s out= 209tok  1. 계약 금액과 매출액을 확인합니다. /    - 계약 금액: 22,764,764,160,000원 /    - 매

Q3 direct  O  gold=2.7        lat= 0.3s out=   4tok  2.7
Q3 cot     O  gold=2.7        lat= 2.7s out= 110tok  1. 먼저 A사와 B사의 설비투자 금액을 확인합니다. /    - A사: 11조 2천억 원 = 11.2조 원 /  

Q4 direct  X  gold=24.6       lat= 0.4s out=   5tok  18.75%
Q4 cot     O  gold=24.6       lat= 3.0s out= 125tok  먼저, 두 기간의 영업이익 차이를 구합니다. /  / - 8조 1천억 원 - 6조 5천억 원 = 1조 6천억 원 /  / 

Q5 direct  X  gold=8년 5개월     lat= 0.5s out=   6tok  9년 5개월
Q5 cot     X  gold=8년 5개월     lat= 3.6s out= 148tok  1. 먼저 전체 기간을 연도와 월로 나누어 계산합니다. /  / 2. 연도 부분: /    - 시작 연도는 2025년입

Q6 direct  O  gold=11.3       lat= 0.4s out=   4tok  11.3
Q6 cot     O  gold=11.3       lat= 2.6s out=  9

---
## 6. function calling 지원 확인

주최 측 참조 아키텍처([CONVENTION §7.6](../docs/CONVENTION.md))가
**function calling · 최대 8스텝 · 인용 강제**를 전제한다.
`tools` 파라미터가 실제로 받아들여지는지 확인한다.

In [7]:
TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'resolve_company',
        'description': '기업 통용명을 DART corp_code와 섹터로 변환한다',
        'parameters': {
            'type': 'object',
            'properties': {'name': {'type': 'string', 'description': '기업 통용명 (예: 현대차)'}},
            'required': ['name'],
        },
    },
}]

r = chat([{'role': 'user', 'content': '현대차의 corp_code를 알려줘. 반드시 도구를 사용해라.'}],
         max_tokens=256, tools=TOOLS, tag='tools')

print('status:', r['status'], '| finish:', r['finish'])
res = (r['raw'] or {}).get('result') or {}
msg = res.get('message') or {}
tc = msg.get('toolCalls') or msg.get('tool_calls')
if r['status'] == 200 and tc:
    print('>>> function calling 지원 확인')
    print(json.dumps(tc, ensure_ascii=False, indent=2)[:800])
elif r['status'] == 200:
    print('>>> 호출은 200이나 toolCalls 없음. 응답:')
    print(msg.get('content', '')[:400])
else:
    print('>>> tools 미지원 또는 스펙 상이. 응답:')
    print(json.dumps(r['raw'], ensure_ascii=False, indent=2)[:800])

status: 200 | finish: tool_calls
>>> function calling 지원 확인
[
  {
    "id": "call_BpYGTnm7BYFvZO8dBAQcfhHy",
    "type": "function",
    "function": {
      "name": "resolve_company",
      "arguments": {
        "name": "현대차"
      }
    }
  }
]


---
## 7. 결과 요약 → 예산 노트북 CONFIG

아래 출력을 [clova_budget.ipynb](clova_budget.ipynb) 의 CONFIG 셀에 반영한다.

> **단가(원/1K 토큰)는 API 응답에 없다.** NCP 요금 페이지에서 확인해
> `PRICE_IN_PER_1K` / `PRICE_OUT_PER_1K` 를 직접 채워야 한다.
> 이 노트북이 확정해 주는 것은 **토큰 수 계수와 지연**이다.

In [8]:
ok = [r for r in CALL_LOG if r['status'] == 200]
tot_in = sum(r['prompt_tokens'] or 0 for r in ok)
tot_out = sum(r['completion_tokens'] or 0 for r in ok)

print('=' * 60)
print('검증 요약')
print('=' * 60)
print(f"엔드포인트        : {ENDPOINT['url']}")
print(f"인증              : {ENDPOINT['auth_name']}")
print(f"모델              : {MODEL}")
print(f"총 호출           : {len(CALL_LOG)}회 (성공 {len(ok)})")
print(f"소비 토큰         : 입력 {tot_in:,} / 출력 {tot_out:,}")
print(f"평균 지연         : {statistics.mean(r['latency_s'] for r in ok):.2f}s")
print()
print('--- clova_budget.ipynb CONFIG 에 반영 ---')
print(f"CHARS_PER_TOKEN_KO = {CHARS_PER_TOKEN_KO:.3f}   # 실측 ({SAMPLE_SRC})")
print("MEASURED['CHARS_PER_TOKEN_KO'] = True")
print()
print('--- 아직 미측정 (NCP 요금 페이지 확인 필요) ---')
print('PRICE_IN_PER_1K    = ?   # 원/입력 1K 토큰')
print('PRICE_OUT_PER_1K   = ?   # 원/출력 1K 토큰')
print()
print('--- 서버 (콘솔 실측값) ---')
print('시간 요금제 92원/h · 월요금제 66,240원 · 공인IP 4,032원/월')
print('손익분기 720h(30일) → 시간 요금제 유리')

# 호출 로그 저장 (재현·감사용)
outp = ROOT / 'notebooks' / 'clova_verify_log.json'
outp.write_text(json.dumps(
    [{k: v for k, v in r.items() if k != 'raw'} for r in CALL_LOG],
    ensure_ascii=False, indent=1), encoding='utf-8')
print()
print('호출 로그 저장:', outp)

검증 요약
엔드포인트        : https://clovastudio.stream.ntruss.com/testapp/v3/chat-completions/HCX-005
인증              : Bearer
모델              : HCX-005
총 호출           : 25회 (성공 25)
소비 토큰         : 입력 1,737 / 출력 2,493
평균 지연         : 2.66s

--- clova_budget.ipynb CONFIG 에 반영 ---
CHARS_PER_TOKEN_KO = 1.722   # 실측 (corpus(20241218800563.xml @0))
MEASURED['CHARS_PER_TOKEN_KO'] = True

--- 아직 미측정 (NCP 요금 페이지 확인 필요) ---
PRICE_IN_PER_1K    = ?   # 원/입력 1K 토큰
PRICE_OUT_PER_1K   = ?   # 원/출력 1K 토큰

--- 서버 (콘솔 실측값) ---
시간 요금제 92원/h · 월요금제 66,240원 · 공인IP 4,032원/월
손익분기 720h(30일) → 시간 요금제 유리

호출 로그 저장: C:\Users\Juhwankim\Documents\Github\ARGUS\notebooks\clova_verify_log.json
